In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

sns.set_theme(style="whitegrid")

DATA_PATH = Path("02_wdi_health_system.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/Users/federico/Desktop/progetto VIS/02_wdi_health_system.csv")
df = pd.read_csv(DATA_PATH)

df = df.dropna(subset=["country_iso3", "year"])
df["year"] = df["year"].astype(int)

INDICATORS = [
    "health_exp_gdp_pct",
    "health_exp_pc_usd",
    "physicians_per_1000",
    "beds_per_1000",
 ]
for col in INDICATORS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

AGGREGATES = {
    "WLD", "HIC", "LIC", "LMC", "UMC", "MIC", "EAP", "EAS", "ECA", "ECS", "LAC", "LCN", "MEA",
    "NAC", "SAS", "SSA", "SSF", "AFW", "AFE", "ARB", "CEB", "EUU", "FCS", "IBD", "IBT", "IDA",
    "IDB", "IDX", "LDC", "LMY", "LTE", "OED", "OSS", "PRE", "PSS", "PST", "SST", "TEA", "TEC",
    "TLA", "TMN", "TSA", "TSS", "UMIC"
}
df_countries = df[~df["country_iso3"].isin(AGGREGATES)].copy()
df_countries = df_countries[df_countries["country_iso3"].str.len() == 3]

norm = df_countries.groupby("year")[INDICATORS].transform(
    lambda s: (s - s.min()) / (s.max() - s.min())
 )
df_countries["health_index"] = norm.mean(axis=1, skipna=True)

df_countries["indicator_count"] = df_countries[INDICATORS].notna().sum(axis=1)
df_countries.loc[df_countries["indicator_count"] < 2, "health_index"] = np.nan

ModuleNotFoundError: No module named 'plotly'

In [ ]:
best_year = (df_countries.groupby("year")["health_index"].apply(lambda s: s.notna().sum())
             .idxmax())
df_latest = df_countries[df_countries["year"] == best_year].dropna(subset=["health_index"])

fig = px.choropleth(
    df_latest, locations="country_iso3", color="health_index", hover_name="country",
    color_continuous_scale=["#F3EAFE", "#D9C7F7", "#B493F0", "#8A5BD6", "#5E2B97"],
    labels={"health_index": "Indice sviluppo sanitario (0-1)"},
    projection="natural earth", locationmode="ISO-3"
)
fig.update_layout(
    title={"x": 0.5, "xanchor": "center", "font": {"size": 18}},
    margin=dict(l=0, r=0, t=30, b=0), template="plotly_white", coloraxis_showscale=False,
    geo=dict(center=dict(lat=20, lon=0), projection_scale=1.1)
)
fig.show()